# NL2Cypher — local experiment runner (Outlines + HF, GPU)

Runs all four settings including `CYPHER_STRICT`, which needs Outlines-constrained
decoding on a local HuggingFace model. **Use an A100 or T4 Colab runtime** — the
notebook pulls torch, transformers and outlines.

**Inputs**
1. A server-produced `bundle_<model>.json` (from `scripts/build_bundle.py`).
2. Bolt access to the Neo4j host that owns the graph.
3. An HF model id or a local checkpoint path.

## 1. Clone the repo and install GPU deps

In [ ]:
%%bash
set -e
if [ ! -d ec3_nl2cypher ]; then
  git clone --depth 1 https://github.com/YOUR_FORK/nl2cypher-on-steroids.git repo
  mv repo/ec3_nl2cypher .
fi
cd ec3_nl2cypher
pip install -q -r requirements-base.txt
# Full LLM stack: torch, transformers, outlines, etc.
pip install -q -r requirements-llm.txt
python -c 'import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else "no gpu")'

In [ ]:
import os, sys
sys.path.insert(0, 'ec3_nl2cypher')

os.environ['NEO4J_URI']       = 'bolt://YOUR_SERVER:7687'
os.environ['NEO4J_USER']      = 'neo4j'
os.environ['NEO4J_PASSWORD']  = 'REPLACE_ME'

os.environ['LLM_PROVIDER']    = 'local'
# Either an HF repo id or a path under /content or Drive.
os.environ['LLM_MODEL_NAME']  = 'Qwen/Qwen2.5-7B-Instruct'

## 2. Load the bundle produced server-side

In [ ]:
import json, pathlib

BUNDLE_PATH = pathlib.Path('bundle_barcelona.json')
bundle = json.loads(BUNDLE_PATH.read_text())
print('bundle keys:', list(bundle))
print('vocab entities:', len(bundle['combined_vocabulary']['entities']))
print('model_dump elements:', len(bundle['model_dump']))

MODEL_DUMP_PATH = pathlib.Path('model_dump.json')
MODEL_DUMP_PATH.write_text(json.dumps(bundle['model_dump']))
print(f'wrote {MODEL_DUMP_PATH} ({MODEL_DUMP_PATH.stat().st_size // 1024} KB)')

## 3. Run all four settings

In [ ]:
from pathlib import Path
from src.config import ExperimentSetting
from src.eval.runner import ExperimentConfig, ExperimentRunner

config = ExperimentConfig(
    name='local_run',
    test_set_path=Path('ec3_nl2cypher/data/test_set.csv'),
    output_dir=Path('results'),
    model_dump_path=MODEL_DUMP_PATH,
    settings=list(ExperimentSetting),  # all 4
)

runner = ExperimentRunner(config=config)
runner.run_comparison(
    test_set_path=config.test_set_path,
    output_dir=config.output_dir,
    settings=config.settings,
)

## 4. Persist results to Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/nl2cypher_results